# 3. League of Legends Global Power Ranking


**PREREQUIST:** It is heavly recommanded that you have prior experience or knowledge

[Global Power Ranking](https://lolesports.com/en-US/news/dev-diary-unveiling-the-global-power-rankings)


[Ranking]() in rstt consists of:
- a [standing](https://rstt.readthedocs.io/en/latest/rstt.html#rstt.Standing): sorts automatically competitors accordingly to their ratings.
- a [datamodel](https://rstt.readthedocs.io/en/latest/rstt.html#rstt.stypes.RatingSystem): container for rating.
- a [backend](https://rstt.readthedocs.io/en/latest/rstt.html#rstt.stypes.Inference): tool to infer ratings, 'the maths'. 
- an [handler](https://rstt.readthedocs.io/en/latest/rstt.html#rstt.stypes.Observer): a workflow handling the update rating based on observations (like game results, or tournament standing).

It looks confusing, mostly because these are rstt notion that you likely have never heard of before. One option is to create a ranking simply by specifying the suited components. For example, take a look at the source code of the [Built-in Elo](https://rstt.readthedocs.io/en/latest/_modules/rstt/ranking/standard/basicElo.html#BasicElo).

```python
class BasicElo(Ranking):
    def __init__(self, name: str, default: float = 1500, k: float = 20.0, lc: float = 400.0,
                 base: float = 10.0, players: list[SPlayer] | None = None):
        super().__init__(name=name,
                         datamodel=KeyModel(default=default),
                         backend=Elo(k=k, lc=lc, base=base),
                         handler=GameByGame(),
                         players=players)]
```

Nothing fancy, simple, reusable class that provide, each, exactly one functionality to the ranking system. For example, appreciate how the Elo backend does not care about the default rating (and does not track any ratings), that is the role of the datamodel.



It is also possible to inherit from the class and override the [forward](https://rstt.readthedocs.io/en/latest/rstt.ranking.html#rstt.ranking.ranking.Ranking.forward) method which allows us to tune the ranking state at update time. The following pseudo-code illustrate the default implementation.

```python
def update(self, *args, **kwargs):
    self.forward(*args, **kwargs) ->
        self.handler.handle_observation(infer=self.backend, datamodel=self.datamodel, *args, **kwargs)
    # automatic ordering of the standing
```

Two important remarks. First the ranking **does not care** about ratings explicitly. It is an implict agreement between the components. As long as the backend can manipulate the ratings provided by the handler (and returned by datamodel.get) it works. It is recommanded to not 'hard code' ratings so that the ranking keeps maximal flexibility and that component remains as independant from each others as possible. Secondly, 'observations' - the things that justify a ranking update, passed as *args **kwargs - are dealt by the handler. The handler needs to extract the relevant information from it. You are free to decide what the ranking takes as input and how to process them.

#### 2.1 Global Power Ranking Logic and Structure

The GPR has its own specificity aswell. It rate leagues, this means we need dedicated Player to represent them because ranking and datamodel are designed for SPlayer. Ratings are evaluated game by game over a time window using an elo rational where games do not all have the same importance. These *extra* notions are develloped in the gpr.utils module.

- (A) 'league' players
- (B) a database
- (C) adjust game importance
- (D) reset ratings (so that only the game in the time period count)
- (E) identify the scene and stage (regional/international) of an event

In the turorial, we use a consistent naming convention (define in the scene module) for competition. This way we can extract the needed information to set the importance of the corresponding game. This mean the GPR will take an 'event: Event' parameter  for update (refered as observation). 



#### Materials

- gpr/utils provides LeagueSystem that tracks Team Region and model League as Player
- gpr/utils provides GameHistory that acts as a simple dataset
- gpr/utils provides an enum class Modes to set the ranking update for teams or regions

If you want to use them, check the following examples. Otherwise feel free to come up with your own solution.

**REMARK:** Regions are names, strings, they are grouped in a StrEnum class Region. League refer to the Player instance that model a region within a ranking.

In [1]:
from rstt import Player, BasicElo, SingleEliminationBracket, LogSolver

from project.gpr.utils import LeagueSystem, GameHistory, Modes
from project.scene import Region


# LeagueSystem
regional_teams: dict[Region: list[Player]] = {}
for region in Region:
    regional_teams[region] = Player.create(nb=16)
ecosystem = LeagueSystem(region_teams=regional_teams)

# GameHistory
ranges: dict[Modes: int] = {Modes.Team: 3, Modes.League: 2}
history = GameHistory(mode_range=ranges)

# access all teams
elo = BasicElo("Ranking", players=ecosystem.teams())


depth = 10
for i in range(depth):
    for region in Region:
        rr = SingleEliminationBracket(f"{region} Season {i}", elo, LogSolver())
        
        # access teams of a given region
        rr.registration(ecosystem.teams(region=region))
        rr.run()

        # register an event in the dataset
        history.add(rr)

# access an event by its name
event = history.get_event("LEC Season 5")
winner = event.top(1)[0]
print(f"{winner.name()} is the winner of {event.name()}")

# get all event in range
events = history.window(Modes.Team)
print([event.name() for event in events])

for event in events:
    elo.update(games=event.games())
number1 = elo[0]
print(f"{number1.name()} is the highest ranked player")

# get the region of a team
reg = ecosystem.region_of(winner)
print(f"{winner.name()} plays in {reg}")

# get the league of a team
league = ecosystem.league_of(ecosystem.region_of(number1))
print(f"{number1.name()} represents {league.name()}")

# You can also get all leagues or regions
print("regions: ", ecosystem.regions())
print("leagues: ", ecosystem.leagues())

Bertha Bell is the winner of LEC Season 5
['LTAN Season 9', 'LTAS Season 9', 'LCP Season 9']
Thomas Banks is the highest ranked player
Bertha Bell plays in LEC
Thomas Banks represents LTAN
regions:  [<Region.LCK: 'LCK'>, <Region.LPL: 'LPL'>, <Region.LEC: 'LEC'>, <Region.LTAN: 'LTAN'>, <Region.LTAS: 'LTAS'>, <Region.LCP: 'LCP'>]
leagues:  [Player - name: LCK, level: -1, Player - name: LPL, level: -1, Player - name: LEC, level: -1, Player - name: LTAN, level: -1, Player - name: LTAS, level: -1, Player - name: LCP, level: -1]


## RatingSystem

Implement the GPR ratings. The key here is to be able to extract ratings of teams and leagues, but also the global power score of teams.

#### TODO:
- make sure you can parametrize your class with x,y and default elo values
- make sure teams and leagues have their respective default elo rating
- implement properly the power score formula in the ordinal() method
- make sure only teams are returned by the keys() method.

**TIPS:** Avoid hard coding a dedicated rating object.
**CHALLENGE:** Idealy Your implementation supports a swap to gaussian ratings as a (mu, sigma) tuple.

In [2]:
from rstt.ranking import KeyModel
from project.gpr import RegionalRatings

# ratings parameters
elo_team    =1500
elo_league  =1000
x = 0.8
y = 0.2

# TODO: Initialise your implementation
gpr_ratings = RegionalRatings(ecosystem=ecosystem,
                              ratings={Modes.Team: KeyModel(default=elo_team),
                                       Modes.League: KeyModel(default=elo_league)})

# check each teams as the default elo
for team in ecosystem.teams():
    
    # TODO: access the team rating
    team_elo = gpr_ratings.ratings[Modes.Team].get(team)
    
    assert team_elo == elo_team, print(team, team_elo)

# check each leagues as the default elo
for league in ecosystem.leagues():
    
    # TODO access the league rating
    league_elo = gpr_ratings.ratings[Modes.League].get(league)
    
    assert league_elo == elo_league, print(team, league_elo)

# check the power score formula
for team in ecosystem.teams():
    assert gpr_ratings.ordinal(team) == (x * elo_team) + y * (elo_league)

# check ratings keys return all teams and no leagues
assert set(gpr_ratings.keys()) == set(ecosystem.teams())


## Elo - The backend

GPR uses Elo formula for rating udpate. There is not much do not here since RSTT provides an implementation [Elo](). The challenge, however, is to adapt the K value based on the event importance. This implies some identification mechanisms. Previously, We defined a naming convention for events that include all pieces of information necessary to tune the ratings update. 

#### Materials:
- project/scheduler/calendar include an "EventInfos" dataclass to encapsulate tournament informations and automatically handle its name.
- project/scheduler/calendar also provides a year_schedule function that return an order list of event information


In [3]:
from project.scheduler.calendar import year_schedule
from project.scene import Split, Finals

start = 2024
end = 2026
events_infos = year_schedule(start, end)

assert len(events_infos) == (end-start) * (len(Region) * len(Split) + len(Finals))

#### TODO:
- Extract an event importance based on its name, e.g "LEC Winter 2025 PlayIns" -> 8
- Adjust the K value of the Elo

In [4]:
from project.gpr.utils import EVENT_NAMING
from project import LeagueSystem, StagedEvent

event = StagedEvent(EVENT_NAMING)

TypeError: StagedEvent.__init__() missing 3 required positional arguments: 'seeding', 'stages', and 'stage_names'